In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
from urllib.parse import urlencode
import nest_asyncio
import uvicorn
import threading
import requests
import pandas as pd
from io import StringIO
import numpy as np
from fastapi.middleware.cors import CORSMiddleware
import yfinance as yf

app = FastAPI()

industry_df = pd.read_csv("openinsider_industries.csv")

app.add_middleware(

    CORSMiddleware,

    allow_origins=["*"],

    allow_credentials=True,

    allow_methods=["*"],

    allow_headers=["*"],

)


class FilterRequest(BaseModel):
    
    sector: str = ""
    subsector: str = ""
    industry: str = ""
    purchase: bool = True
    sale: bool = False
    award: bool = False

    ceo: bool = False
    cfo: bool = False
        
    ticker: str=""

    min_value:int = 1000000
    
    min_marketcap: float = 0

    days:int = 30

In [2]:
@app.get("/industries")
def get_industries():

    return industry_df.to_dict(orient="records")



@app.post("/search")
def search(req: FilterRequest):

    filters = req.model_dump()
    
    print(filters)

    df = get_insider_records(filters)

    df = add_market_cap(df)

    mincap = filters.get("min_marketcap", 0)

    if mincap > 0:

        df = df[df["MarketCap"] >= mincap * 1e9]


    print("NaN count:", df.isna().sum().sum())

    print("Inf count:",np.isinf(df.select_dtypes(include=[np.number])).sum().sum())
    
    df = df.replace([np.inf, -np.inf], np.nan)

    df = df.fillna("")

    return df.to_dict(orient="records")

In [3]:
def find(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    r = requests.get(url, headers=headers, timeout=100)
    #print(r.status_code)
    tables = pd.read_html(StringIO(r.text))
    for table in tables:
        cols = [str(c).replace("\xa0", " ").strip() for c in table.columns]
        #cols = " ".join(table.columns.astype(str))
        if "Ticker" in cols and "Value" in cols:
            return table
    return pd.DataFrame()


In [4]:
def build_url(filters):
    
    print("***** NEW BUILD_URL *****")
    print(filters)

    base = "http://openinsider.com/screener"

    params = {
        
        # Ticker
        "s": filters.get("ticker",""),

        # Transaction type
        "xp": int(filters.get("purchase", False)),
        "xs": int(filters.get("sale", False)),
        "xa": int(filters.get("award", False)),


        # Insider position
        "isceo": int(filters.get("ceo", False)),
        "iscfo": int(filters.get("cfo", False)),


        # Transaction value
        "vl": filters.get("min_value", ""),


        # Filing period
        "fd": filters.get("days", 730),


        # Output size
        "cnt": 100,
        "page": 1
    }
    
    # Industry filtering
    industry = filters.get("industry", "")
    subsector = filters.get("subsector", "")
    sector = filters.get("sector", "")
    
    rows = None

    if industry:
        rows = industry_df[industry_df["sic3"] == int(industry)]

    elif subsector:
        rows = industry_df[industry_df["sic2"] == int(subsector)]

    elif sector:
        rows = industry_df[industry_df["sic1"] == int(sector)]

    if rows is not None and not rows.empty:
        params["sicl"] = int(rows["sicl"].min())
        params["sich"] = int(rows["sich"].max())
    
    return base + "?" + urlencode(params)

In [5]:
def get_insider_records(filters):

    url = build_url(filters)

    print("URL:")
    print(url)

    df = find(url)

    if df is None:
        return pd.DataFrame()

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    df = df.rename(columns={

        "Trade\xa0Type": "Trade Type",
        "Company\xa0Name": "Company Name",
        "Filing\xa0Date": "Filing Date",
        "Insider\xa0Name": "Insider Name",
        "Trade\xa0Date": "Trade Date"

    })

    return df

In [6]:
def add_market_cap(df):
    
    if df.empty:
        print("No records found.")
        return df
    
    if "Ticker" not in df.columns:

        print("Ticker column missing!")
        print(df.columns.tolist())
        return df


    marketcaps = []

    for ticker in df["Ticker"]:

        try:

            cap = yf.Ticker(ticker).fast_info.get("marketCap")

        except Exception:

            cap = None

        marketcaps.append(cap)

    df["MarketCap"] = marketcaps

    return df

In [7]:
nest_asyncio.apply()

def run_server():
    
    uvicorn.run(app, host="127.0.0.1", port=8001)

thread = threading.Thread(target=run_server, daemon=True)

thread.start()

INFO:     Started server process [40192]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


Test Connection

In [8]:
response = requests.post(
    "http://127.0.0.1:8001/search",
    json={
        "min_marketcap": 0,
        "sector": "",
        "subsector": "",
        "industry": "",
        'ticker': 'JEF',
        "purchase": True,
        "sale": True,
        "award": True,
        "ceo": False,
        "cfo": False,
        "min_value": 10000,
        "days": 30
    }
)

print(response.status_code)
#print(response.text)

{'sector': '', 'subsector': '', 'industry': '', 'purchase': True, 'sale': True, 'award': True, 'ceo': False, 'cfo': False, 'ticker': 'JEF', 'min_value': 10000, 'min_marketcap': 0.0, 'days': 30}
***** NEW BUILD_URL *****
{'sector': '', 'subsector': '', 'industry': '', 'purchase': True, 'sale': True, 'award': True, 'ceo': False, 'cfo': False, 'ticker': 'JEF', 'min_value': 10000, 'min_marketcap': 0.0, 'days': 30}
URL:
http://openinsider.com/screener?s=JEF&xp=1&xs=1&xa=1&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1
Shape: (1, 16)
Columns: ['X', 'Filing\xa0Date', 'Trade\xa0Date', 'Ticker', 'Insider\xa0Name', 'Title', 'Trade\xa0Type', 'Price', 'Qty', 'Owned', 'ΔOwn', 'Value', '1d', '1w', '1m', '6m']
NaN count: 5
Inf count: 0
INFO:     127.0.0.1:59881 - "POST /search HTTP/1.1" 200 OK
200
INFO:     127.0.0.1:59905 - "OPTIONS /search HTTP/1.1" 200 OK
{'sector': '15', 'subsector': '16', 'industry': '', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'm

ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'MarketCap'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\uvicorn\protocols\http\h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages

INFO:     127.0.0.1:59906 - "GET /industries HTTP/1.1" 200 OK
INFO:     127.0.0.1:59944 - "OPTIONS /search HTTP/1.1" 200 OK
{'sector': '10', 'subsector': '12', 'industry': '1221', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 5.0, 'days': 30}
***** NEW BUILD_URL *****
{'sector': '10', 'subsector': '12', 'industry': '1221', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 5.0, 'days': 30}
URL:
http://openinsider.com/screener?s=&xp=1&xs=0&xa=0&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1&sicl=1221&sich=1310
Shape: (0, 0)
Columns: []
No records found.
INFO:     127.0.0.1:59944 - "POST /search HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'MarketCap'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\uvicorn\protocols\http\h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages

INFO:     127.0.0.1:59945 - "OPTIONS /search HTTP/1.1" 200 OK
{'sector': '10', 'subsector': '12', 'industry': '1221', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 5.0, 'days': 30}
***** NEW BUILD_URL *****
{'sector': '10', 'subsector': '12', 'industry': '1221', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 5.0, 'days': 30}
URL:
http://openinsider.com/screener?s=&xp=1&xs=0&xa=0&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1&sicl=1221&sich=1310
Shape: (0, 0)
Columns: []
No records found.
INFO:     127.0.0.1:59945 - "POST /search HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'MarketCap'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages\uvicorn\protocols\http\h11_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
  File "C:\Users\25280\AppData\Roaming\Python\Python39\site-packages

INFO:     127.0.0.1:52446 - "GET /industries HTTP/1.1" 200 OK
INFO:     127.0.0.1:59875 - "OPTIONS /search HTTP/1.1" 200 OK
{'sector': '', 'subsector': '', 'industry': '', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 0.0, 'days': 30}
***** NEW BUILD_URL *****
{'sector': '', 'subsector': '', 'industry': '', 'purchase': True, 'sale': False, 'award': False, 'ceo': False, 'cfo': False, 'ticker': '', 'min_value': 10000, 'min_marketcap': 0.0, 'days': 30}
URL:
http://openinsider.com/screener?s=&xp=1&xs=0&xa=0&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1
Shape: (20, 17)
Columns: ['X', 'Filing\xa0Date', 'Trade\xa0Date', 'Ticker', 'Company\xa0Name', 'Insider\xa0Name', 'Title', 'Trade\xa0Type', 'Price', 'Qty', 'Owned', 'ΔOwn', 'Value', '1d', '1w', '1m', '6m']
NaN count: 89
Inf count: 0
INFO:     127.0.0.1:59875 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:59876 - "OPTIONS /search HTTP/1.1" 200 OK
{'sector'

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: GLDD"}}}
$GLDD: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$GLDD: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")
$GLDD: possibly delisted; no price data found  (period=1y) (Yahoo error = "No data found, symbol may be delisted")
$GLDD: possibly delisted; no price data found  (period=5d) (Yahoo error = "No data found, symbol may be delisted")


NaN count: 313
Inf count: 0
INFO:     127.0.0.1:56684 - "POST /search HTTP/1.1" 200 OK


In [8]:
data = response.json()
df_result = pd.DataFrame(data)
df_result

,X,Filing Date,Trade Date,Ticker,Company Name,Insider Name,Title,Trade Type,Price,Qty,Owned,ΔOwn,Value,1d,1w,1m,6m
0,D,2026-07-21 21:16:30,2026-07-21,UAL,"United Airlines Holdings, Inc.",Kirby J Scott,CEO,S - Sale+OE,$118.05,-159321,810851,-16%,"-$18,807,047",,,,
1,M,2026-07-21 19:52:05,2026-07-17,BFLY,"Butterfly Network, Inc.",Rothberg Jonathan M,"Dir, 10%",S - Sale,$6.64,-3126453,22919427,-12%,"-$20,748,309",,,,
2,D,2026-07-21 16:47:15,2026-07-20,TDG,Transdigm Group Inc,Howley W Nicholas,Dir,S - Sale+OE,"$1,216.08",-10132,21548,-32%,"-$12,321,288",,,,
3,,2026-07-20 21:27:45,2026-07-17,SN,"Sharkninja, Inc.",Barrocas Mark,CEO,S - Sale,$155.01,-250000,1996659,-11%,"-$38,752,500",,,,
4,D,2026-07-20 17:30:09,2026-07-16,CBIO,"Crescent Biopharma, Inc.",Fairmount Funds Management LLC,"Dir, 10%",P - Purchase,$14.50,853450,3601316,+31%,"+$12,375,025",,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,D,2026-07-02 16:15:17,2026-07-02,LIME,"Neutron Holdings, Inc.","Uber Technologies, Inc",10%,P - Purchase,$25.00,800000,14859661,+6%,"+$20,000,000",,,,
96,,2026-07-02 16:06:41,2026-07-01,OKLO,Oklo Inc.,Dewitte Jacob,"Co-Founder, CEO, 10%",S - Sale,$52.93,-200000,20511338,-1%,"-$10,585,600",,,,
97,,2026-07-02 16:05:08,2026-07-01,OKLO,Oklo Inc.,Cochran Caroline,"Co-Founder, COO, 10%",S - Sale,$52.93,-200000,20511338,-1%,"-$10,585,600",,,,
98,,2026-07-02 12:18:12,2026-06-30,AVAH,"Aveanna Healthcare Holdings, Inc.",Williams Robert M Jr,10%,S - Sale,$8.01,-3000000,29065892,-9%,"-$24,030,000",,,,


INFO:     127.0.0.1:54945 - "OPTIONS /search HTTP/1.1" 200 OK
URL:
http://openinsider.com/screener?xp=1&xs=0&xa=0&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1
Shape:
(23, 17)
X                object
Filing Date      object
Trade Date       object
Ticker           object
Company Name     object
Insider Name     object
Title            object
Trade Type       object
Price            object
Qty               int64
Owned             int64
ΔOwn             object
Value            object
1d              float64
1w              float64
1m              float64
6m              float64
dtype: object
NaN count: 103
Inf count: 0
INFO:     127.0.0.1:54945 - "POST /search HTTP/1.1" 200 OK
INFO:     127.0.0.1:59952 - "OPTIONS /search HTTP/1.1" 200 OK
URL:
http://openinsider.com/screener?xp=1&xs=0&xa=0&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1
Shape:
(23, 17)
X                object
Filing Date      object
Trade Date       object
Ticker           object
Company Name     object
Insider Name     ob

In [27]:
df_result = df_result.rename(columns={"Trade\xa0Type":"Trade Type", "Company\xa0Name": "Company Name", "Filing\xa0Date": "Filing Date", \
                        "Insider\xa0Name": "Insider Name", "Trade\xa0Date":"Trade Date"})

In [24]:
response = requests.post(
    "http://127.0.0.1:8000/search",
    json={
        "Ticker": "JEF"
        "purchase": True,
        "sale": True,
        "award": True,
        "ceo": False,
        "cfo": False,
        "min_value": 10000,
        "days": 30
    }
)

data = response.json()

print(data[0])

URL:
http://openinsider.com/screener?xp=1&xs=1&xa=1&isceo=0&iscfo=0&vl=10000&fd=30&cnt=100&page=1
Shape:
(100, 17)
X                object
Filing Date      object
Trade Date       object
Ticker           object
Company Name     object
Insider Name     object
Title            object
Trade Type       object
Price            object
Qty               int64
Owned             int64
ΔOwn             object
Value            object
1d              float64
1w              float64
1m              float64
6m              float64
dtype: object
NaN count: 438
Inf count: 0
INFO:     127.0.0.1:49268 - "POST /search HTTP/1.1" 200 OK
{'X': '', 'Filing\xa0Date': '2026-07-20 21:27:45', 'Trade\xa0Date': '2026-07-17', 'Ticker': 'SN', 'Company\xa0Name': 'Sharkninja, Inc.', 'Insider\xa0Name': 'Barrocas Mark', 'Title': 'CEO', 'Trade\xa0Type': 'S - Sale', 'Price': '$155.01', 'Qty': -250000, 'Owned': 1996659, 'ΔOwn': '-11%', 'Value': '-$38,752,500', '1d': '', '1w': '', '1m': '', '6m': ''}


Build parser for industry, no need to run every time

In [53]:
import re
import ast

def extract_array(var_name, js_text):
    """
    Extract a JavaScript array from sic.js and convert it to a Python object.

    Parameters
    ----------
    var_name : str
        Variable name, e.g. "SECTOR_CODES"
    js_text : str
        Entire contents of sic.js

    Returns
    -------
    list
        Python list (possibly nested)
    """

    pattern = rf'var\s+{re.escape(var_name)}\s*=\s*(\[[\s\S]*?\]);'

    m = re.search(pattern, js_text)

    if m is None:
        raise ValueError(f"{var_name} not found")

    array_text = m.group(1)

    # JavaScript arrays are already almost valid Python lists
    return ast.literal_eval(array_text)

In [54]:
with open("sic.js", "r", encoding="utf-8") as f:
    js = f.read()

SECTOR_CODES      = extract_array("SECTOR_CODES", js)
SECTOR_CODES_MAX  = extract_array("SECTOR_CODES_MAX", js)
SECTOR_NAMES      = extract_array("SECTOR_NAMES", js)

SUBSECTOR_CODES   = extract_array("SUBSECTOR_CODES", js)
SUBSECTOR_CODES2  = extract_array("SUBSECTOR_CODES2", js)
SUBSECTOR_NAMES   = extract_array("SUBSECTOR_NAMES", js)
SUBSECTOR_NAMES2  = extract_array("SUBSECTOR_NAMES2", js)

INDUSTRY_CODES    = extract_array("INDUSTRY_CODES", js)
INDUSTRY_NAMES    = extract_array("INDUSTRY_NAMES", js)

In [55]:
sector_lookup = dict(zip(SECTOR_CODES, SECTOR_NAMES))

subsector_lookup = {}
subsector_to_sector = {}

for sector, codes, names in zip(
        SECTOR_CODES,
        SUBSECTOR_CODES,
        SUBSECTOR_NAMES):

    for code, name in zip(codes, names):

        subsector_lookup[code] = name
        subsector_to_sector[code] = sector

In [56]:
sector_lookup = dict(zip(SECTOR_CODES, SECTOR_NAMES))

subsector_lookup = {}
subsector_to_sector = {}

for sector, codes, names in zip(
        SECTOR_CODES,
        SUBSECTOR_CODES,
        SUBSECTOR_NAMES):

    for code, name in zip(codes, names):

        subsector_lookup[code] = name
        subsector_to_sector[code] = sector

In [57]:
def industries_of_subsector(subsector):

    lower = subsector * 100

    lo = None
    hi = None

    for i, code in enumerate(INDUSTRY_CODES):

        if code >= lower and lo is None:
            lo = i

        if code - 100 >= lower:
            hi = i
            break

    if lo is None:
        return []

    if hi is None:
        hi = len(INDUSTRY_CODES)

    return list(range(lo, hi))

In [58]:
def industries_of_subsector(subsector):

    lower = subsector * 100

    lo = None
    hi = None

    for i, code in enumerate(INDUSTRY_CODES):

        if code >= lower and lo is None:
            lo = i

        if code - 100 >= lower:
            hi = i
            break

    if lo is None:
        return []

    if hi is None:
        hi = len(INDUSTRY_CODES)

    return list(range(lo, hi))

In [59]:
def industry_range(industry):

    idx = INDUSTRY_CODES.index(industry)

    sicl = industry

    if idx == len(INDUSTRY_CODES)-1:
        sich = 9999
    else:
        sich = INDUSTRY_CODES[idx+1]-1

    return sicl, sich

In [60]:
rows = []

for code, name in zip(INDUSTRY_CODES[1:], INDUSTRY_NAMES[1:]):

    # -------------------------------------------------
    # Determine subsector
    # -------------------------------------------------

    if code in (8880, 8888):
        subsector = 88
    elif code == 9721:
        subsector = 97
    elif code == 9995:
        subsector = 99
    else:
        subsector = code // 100

    # -------------------------------------------------
    # Determine sector
    # -------------------------------------------------

    if subsector in subsector_to_sector:
        sector = subsector_to_sector[subsector]
    else:
        # Find the nearest previous valid subsector
        prev = max(
            s for s in subsector_to_sector
            if s <= subsector
        )
        sector = subsector_to_sector[prev]

    # -------------------------------------------------
    # Subsector name
    # -------------------------------------------------

    subsector_name = subsector_lookup.get(subsector, "Other")

    # -------------------------------------------------
    # SIC range
    # -------------------------------------------------

    idx = INDUSTRY_CODES.index(code)

    sicl = code

    if idx == len(INDUSTRY_CODES)-1:
        sich = 9999
    else:
        sich = INDUSTRY_CODES[idx+1]-1

    rows.append({
        "sic1": sector,
        "sector_name": sector_lookup[sector],

        "sic2": subsector,
        "subsector_name": subsector_name,

        "sic3": code,
        "industry_name": name,

        "sicl": sicl,
        "sich": sich
    })

df = pd.DataFrame(rows)

In [61]:
import pandas as pd

df = pd.DataFrame(rows)

df.drop_duplicates(
    subset="sic3",
    inplace=True
)

df.sort_values(
    ["sic1","sic2","sic3"],
    inplace=True
)

In [62]:
print(len(df))

444


In [63]:
df.to_csv(
    "openinsider_industries.csv",
    index=False
)

In [64]:
all_codes = set(INDUSTRY_CODES)
all_codes.remove(0)

parsed_codes = set(df["sic3"])

missing = sorted(all_codes - parsed_codes)

print(len(missing))
print(missing)

0
[]


In [65]:
print(type(SECTOR_CODES))
print(type(SUBSECTOR_CODES))
print(type(SUBSECTOR_CODES[0]))
print(type(INDUSTRY_NAMES))

print(SECTOR_CODES[:5])
print(SUBSECTOR_CODES[:3])
print(INDUSTRY_CODES[:5])

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
[-1, 1, 10, 15, 20]
[[0], [0, 1, 2, 7, 8, 9], [0, 10, 12, 13, 14]]
[0, 100, 200, 700, 800]
